# 03 — Preserve useful alternatives, then prove the result

Let's make a small frontier fail honestly rather than silently returning an incomplete answer. Then we'll inspect the independent oracle and a few request-bound failures.

The frontier exercise is separate from the scalar routing engine. It is **not** an implementation of full McRAPTOR or a passenger recommendation policy. Read [chapter 07](../docs/07_multicriteria_frontiers_and_extensions.md).

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src" / "raptor.py").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from src import compile_timetable, demo_timetable, raptor, RoutingError
from src.pareto import Objective, bounded_frontier, dominates
fast = Objective("fast", departure=0, arrival=100, boardings=1, walking=30, access_burden=0, minimum_slack=30)
less_walk = Objective("less-walk", departure=0, arrival=110, boardings=1, walking=10, access_burden=0, minimum_slack=30)
assert not dominates(fast, less_walk)
assert not dominates(less_walk, fast)
frontier = bounded_frontier((fast, less_walk), capacity=2)
print("Preserved:", [item.journey_id for item in frontier])

## Lose a real path at an intermediate stop
Run the actual scalar router on the walking-tradeoff network. At M, arrival 20 replaces arrival 21 after one boarding. The explicitly enumerated low-walking journey is still feasible, but cannot be recovered by sorting the scalar destination results. Both witnesses are checked against raw input.
This is a counterexample, not full McRAPTOR or a multicriteria oracle. See [the worked network](../docs/07_multicriteria_frontiers_and_extensions.md#run-a-path-loss-counterexample).

In [ ]:
from src.fixtures import walking_tradeoff_timetable
from example_walking_tradeoff import low_walking_witness
tradeoff = walking_tradeoff_timetable()
scalar = raptor(compile_timetable(tradeoff), "O", "Z", 0, max_boardings=2)
fast_path, = scalar.journeys()
low_path = low_walking_witness()
assert scalar.rows[1]["M"].time == 20
for label, path in (("scalar", fast_path), ("lost but feasible", low_path)):
    path.validate_against(tradeoff)
    print(label, path.arrival, path.boardings, path.walking_seconds)
assert (fast_path.arrival, fast_path.boardings, fast_path.walking_seconds) == (30, 2, 8)
assert (low_path.arrival, low_path.boardings, low_path.walking_seconds) == (35, 2, 1)
assert low_path not in scalar.journeys()


## Exhaust capacity without lying

A public result cap isn't an internal frontier policy. With capacity one, this exercise refuses to lose either nondominated objective vector.

In [ ]:
try:
    bounded_frontier((fast, less_walk), capacity=1)
except RoutingError as error:
    assert error.code == "RAPTOR_FRONTIER_CAPACITY_EXCEEDED"
    print(error.code)
else:
    raise AssertionError("The frontier silently lost a required candidate.")

## A dominated candidate should disappear safely

The slow-and-long-walking candidate is worse than `fast` on both changed criteria, with no compensating advantage. Removing it is a dominance decision, not arbitrary truncation.

In [ ]:
dominated = Objective("dominated", departure=0, arrival=120, boardings=1, walking=40, access_burden=0, minimum_slack=30)
assert dominates(fast, dominated)
assert bounded_frontier((fast, dominated), capacity=1) == (fast,)
print("Dominated candidate removed; the useful candidate remains.")

## Compare every stop, not just the winning card

The routing oracle uses `(stop, exact_boardings)` states and enumerates all legal board/alight possibilities. The comparison below checks every stop at every at-most boarding budget.

In [ ]:
from src.oracle import oracle_arrivals
index = compile_timetable(demo_timetable())
result = raptor(index, "O", "Z", 8 * 3600, max_boardings=3, boarding_slack=60)
expected = oracle_arrivals(index.timetable, "O", 8 * 3600, max_boardings=3, boarding_slack=60)
actual = tuple({stop: label.time for stop, label in row.items()} for row in result.rows)
assert actual == expected
print("All stop/budget labels agree with the independent oracle.")

## A valid shape isn't a valid generation

The fake bundle ID below exists only to illustrate identity matching. A stale or mismatched snapshot must not be replaced by the last successful result. This toy overlay does not verify signatures or act as a realtime provider.

In [ ]:
from src.realtime import Snapshot, apply_snapshot
snapshot = Snapshot("toy-rt", "a" * 64, index.timetable.service_date, observed_at=100, valid_until=200)
try:
    apply_snapshot(index.timetable, snapshot, now=200)
except RoutingError as error:
    assert error.code == "REALTIME_STALE"
    print(error.code)
else:
    raise AssertionError("An expired snapshot was accepted.")

## Budget failures are observable outcomes

We only report work performed in this lab. No synthetic provider-call zeros or production latency claims are added to the metrics.

In [ ]:
from src.metrics import Work
try:
    raptor(index, "O", "Z", 8 * 3600, work=Work(max_units=1))
except RoutingError as error:
    assert error.code == "RAPTOR_WORK_CAPACITY_EXCEEDED"
    print(error.code)
else:
    raise AssertionError("The request exceeded its budget without failing.")
print("Measured successful-query operations:", result.metrics)

## The next real extension

To build full multicriteria routing, construct a counterexample where a useful walking/slack label is lost at an intermediate stop. Add an independent unbounded multicriteria oracle, then introduce the corresponding state bag and versioned capacity policy.

This notebook verifies the local model only. Full multicriteria routing would need intermediate label bags and an independent multicriteria oracle. Multi-date input, signed data verification, and a deployed service are outside this implementation.